In [151]:
import numpy as np
import pyvista as pv
from scipy.spatial import KDTree

case_name = "case1"
mesh_name = "coil_box"

sol_ngsolve = pv.read(f"output/{case_name}/{case_name}_ngsolve.vtu")
B_norm_ngsolve = sol_ngsolve["Magnetic_flux_density_norm"]
B_real_ngsolve = sol_ngsolve["B"]

sol_comsol = pv.read(f"output/{case_name}/{case_name}_comsol.vtu")
B_norm_comsol = sol_comsol["Magnetic_flux_density_norm"]
B_x_real_comsol = sol_comsol["B_x"][:, np.newaxis]
B_y_real_comsol = sol_comsol["B_y"][:, np.newaxis]
B_z_real_comsol = sol_comsol["B_z"][:, np.newaxis]

print(B_norm_ngsolve.shape)
print(B_norm_comsol.shape)

points_ngsolve = sol_ngsolve.points
points_comsol = sol_comsol.points

tree2 = KDTree(points_comsol)
_, indices = tree2.query(points_ngsolve)
B_norm_comsol = B_norm_comsol[indices]
B_x_real_comsol = B_x_real_comsol[indices]
B_y_real_comsol = B_y_real_comsol[indices]
B_z_real_comsol = B_z_real_comsol[indices]
B_real_comsol = np.concatenate((B_x_real_comsol, B_y_real_comsol, B_z_real_comsol), axis=1)

(12090,)
(12090,)


In [152]:
mesh = pv.read(f"meshes/coil_box.msh")

In [153]:
# NGSolve and COMSOL

abs_error = np.abs(B_norm_ngsolve - B_norm_comsol) * 100 / np.abs(B_norm_comsol)

abs_error_max = np.max(abs_error)
abs_error_mean = np.mean(abs_error)

print (f"Case: {case_name}")
print("")
print(f"  * Max. absolute error between NGSolve and COMSOL : {abs_error_max:.3e}.")
print(f"  * Avg. absolute error between NGSolve and COMSOL : {abs_error_mean:.3e}.")
print("")

mesh["relative_error_ngsolve_comsol"] = abs_error

abs_error = np.abs(B_real_ngsolve[:, 0] - B_real_comsol[:, 0]) / np.abs(B_real_comsol[:, 0])

abs_error_max = np.max(abs_error)
abs_error_mean = np.mean(abs_error)

print (f"Case: {case_name}")
print("")
print(f"  * Max. absolute error between NGSolve and COMSOL : {abs_error_max:.3e}.")
print(f"  * Avg. absolute error between NGSolve and COMSOL : {abs_error_mean:.3e}.")
print("")

# abs_error = np.abs(B_real_ngsolve[:, 1] - B_real_comsol[:, 1]) / np.abs(B_real_comsol[:, 1])
# abs_error[np.isinf(abs_error)] = np.nan

# abs_error_max = np.nanmax(abs_error)
# abs_error_mean = np.nanmean(abs_error)

# print (f"Case: {case_name}")
# print("")
# print(f"  * Max. absolute error between NGSolve and COMSOL : {abs_error_max:.3e}.")
# print(f"  * Avg. absolute error between NGSolve and COMSOL : {abs_error_mean:.3e}.")
# print("")

# abs_error = np.abs(B_real_ngsolve[:, 2] - B_real_comsol[:, 2]) / np.abs(B_real_comsol[:, 2])

# abs_error_max = np.max(abs_error)
# abs_error_mean = np.mean(abs_error)

# print (f"Case: {case_name}")
# print("")
# print(f"  * Max. absolute error between NGSolve and COMSOL : {abs_error_max:.3e}.")
# print(f"  * Avg. absolute error between NGSolve and COMSOL : {abs_error_mean:.3e}.")
# print("")

# abs_error = np.abs(points_ngsolve - points_comsol[indices, :])

# abs_error_max = np.nanmax(abs_error)
# abs_error_mean = np.nanmean(abs_error)

# print (f"Case: {case_name}")
# print("")
# print(f"  * Max. absolute error between NGSolve and COMSOL : {abs_error_max:.3e}.")
# print(f"  * Avg. absolute error between NGSolve and COMSOL : {abs_error_mean:.3e}.")
# print("")

Case: case1

  * Max. absolute error between NGSolve and COMSOL : 9.514e+01.
  * Avg. absolute error between NGSolve and COMSOL : 4.467e+00.

Case: case1

  * Max. absolute error between NGSolve and COMSOL : 1.852e+02.
  * Avg. absolute error between NGSolve and COMSOL : 2.010e+00.



In [154]:
mesh.save(f"output/{case_name}/{case_name}_error_ngsolve_comsol.vtu")
